Что такое immutable класс?
- это класс, экземпляры которого нельзя изменить после создания. Если нужно изменить объект такого класса, создаётся новый объект с обновлёнными значениями
- обычно используются как структура для хранения данных, без логики обработки 
- примеры: tuple, str

In [1]:
s = "hello"
print(id(s))  # адрес в памяти

# Попытка "изменить" строку
s += " world"
print(s)      # "hello world"
print(id(s))  # другой адрес

4581120496
hello world
4584924144


Для начала, посмотрим, как происходит создание объекта в Python. На самом деле перед `__init__` вызывается `__new__` - именно он выделяет память и возвращает объект класса

In [2]:
class MyClass:
    def __new__(cls, *args, **kwargs):
        print("1. __new__ вызван - создаём объект")
        instance = super().__new__(cls)  # Создаём экземпляр
        print(f"2. Объект создан: {id(instance)}")
        return instance  # Возвращаем созданный объект
    
    def __init__(self, name):
        print("3. __init__ вызван - инициализируем объект")
        print(f"4. Инициализируем объект: {id(self)}")
        self.name = name
        print("5. Инициализация завершена")
        # __init__ не возвращает ничего (None)

# Создаём объект
obj = MyClass("test")
print(f"Итоговый объект: {id(obj)}")

1. __new__ вызван - создаём объект
2. Объект создан: 4585163264
3. __init__ вызван - инициализируем объект
4. Инициализируем объект: 4585163264
5. Инициализация завершена
Итоговый объект: 4585163264


In [3]:
# откроем исходники (ctrl + click)
tuple.__new__(tuple, [1, 2, 3])  # __new__ выделяет память, создаёт объект, используется перед __init__. У tuple он принимает итератор объектов, заполняется ими
tuple.__init__(tuple)  # не определён, tuple наследует __init__ от object
# tuple иммутабельный, его нельзя изменить (в том числе инициализировать) после создания

In [4]:
# а ещё __new__ хорош, когда надо определить метод копирования, когда мы хотим создать "голый" объект класса без атрибутов

class Person:
    def __init__(self, name, age, city):
        self.name = name
        self.age = age
        self.city = city
    
    def copy(self):
        new_instance = self.__class__.__new__(self.__class__)  # обязательно используйте self.__class__, а не Person - иначе будет факап в наследниках (они же не Person)
        new_instance.name = self.name
        new_instance.age = self.age
        new_instance.city = self.city
        return new_instance

#### Способ 1

Самый популярный способ сделать immutable - dataclass

In [5]:
from dataclasses import dataclass

@dataclass(frozen=True)
class ImmutablePoint:
    x: int
    y: int

point = ImmutablePoint(10, 'kek')
display(point)
point.x = 11

ImmutablePoint(x=10, y='kek')

FrozenInstanceError: cannot assign to field 'x'

В идеале - pydantic, который проверит типы атрибутов при инициализации

In [6]:
from pydantic import BaseModel, ConfigDict

class ImmutablePoint(BaseModel):
    model_config = ConfigDict(frozen=True)
    x: int
    y: int
point = ImmutablePoint(x=10, y=20)
display(point)
point = ImmutablePoint(x=10, y='kek')

ImmutablePoint(x=10, y=20)

ValidationError: 1 validation error for ImmutablePoint
y
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='kek', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing

#### Способ 2

Наследуемся от иммутабельной коллекции, оборачиваем атрибуты в property 

In [8]:
class ImmutablePoint(tuple):  # лучше наследоваться от иммутабельной коллекции, чтобы нельзя было поменять атрибуты через super()
    def __new__(cls, *args):
        return super().__new__(cls, (args))
    
    @property
    def x(self):
        return self[0]
    
    @property
    def y(self):
        return self[1]

point = ImmutablePoint(10, 20)
display(point)
point.x = 11

(10, 20)

AttributeError: can't set attribute 'x'

#### Способ 3

Заполнить атрибуты вручную. Запретить обращение к атрибутам

In [9]:
class ImmutablePoint:
    def __new__(cls, x, y):
        instance = super().__new__(cls)
        # Теперь заполняем атрибуты для него
        # Используем super().__setattr__ чтобы обойти наш __setattr__
        super(ImmutablePoint, instance).__setattr__('x', x)  # указываем класс, для которого ищем __setattr__ и конкретный объект instance, для которого вызываем
        super(ImmutablePoint, instance).__setattr__('y', y)
        return instance
    
    def __setattr__(self, name, value):
        # Если атрибут уже существует, запрещаем его изменение
        if hasattr(self, name):
            raise AttributeError(f"Cannot modify attribute '{name}' of immutable object")

        # Если атрибута нет, тоже запрещаем (объект уже создан)
        raise AttributeError(f"Cannot add new attribute '{name}' to immutable object")
    
    def __repr__(self):  # стоит переопределить, мы не наследуем этот метод
        return f"ImmutablePoint(x={self.x}, y={self.y})"

point = ImmutablePoint(10, 20)
display(point)
point.x = 11

ImmutablePoint(x=10, y=20)

AttributeError: Cannot modify attribute 'x' of immutable object